In [1]:
# 1. Connect to Snowflake and load the tables (saves a local copy)
import os
import pandas as pd
import numpy as np
from IPython.display import display

TABLES = ["LOANS", "CUSTOMERS", "PAYMENTS"]
data = {}

if all(os.path.exists(f"{t.lower()}.pkl") for t in TABLES):
    
    data = {t.lower(): pd.read_pickle(f"{t.lower()}.pkl") for t in TABLES}
    print("Loaded from saved .pkl files")
else:
    import snowflake.connector
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))  # finds Oriel_project-/.env from notebooks/
    conn = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        user=os.environ["SNOWFLAKE_USER"],
        password=os.environ["SNOWFLAKE_PASSWORD"],
        warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
        database=os.environ["SNOWFLAKE_DATABASE"],
        schema=os.environ["SNOWFLAKE_SCHEMA"],
    )
    cur = conn.cursor()
    for t in TABLES:
        cur.execute(f"SELECT * FROM ORIEL_COLLECTIONS.PUBLIC.{t}")
        df = cur.fetch_pandas_all()
        df.columns = df.columns.str.lower()
        df = df[[c for c in df.columns if not c.startswith("_a")]]  # drop Airbyte columns
        df.to_pickle(f"{t.lower()}.pkl")
        data[t.lower()] = df
        print(t, df.shape)


c:\Users\HomePC\Documents\Amdari-Projects\september-cohorts\nexora-project\venv\Lib\site-packages\snowflake\connector\connection\_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


LOANS (42168, 12)
CUSTOMERS (18360, 13)
PAYMENTS (46506, 8)


In [3]:
# 2. Clean loans: types, dates, status, duplicates, "repaid on time"
loans = data["loans"].copy()
for c in ["term_days", "days_past_due", "total_repayable_gbp", "principal_amount_gbp"]:
    loans[c] = pd.to_numeric(loans[c], errors="coerce")
for c in ["disbursement_date", "due_date", "closed_date"]:
    loans[c] = pd.to_datetime(loans[c], errors="coerce")

loans["status"] = loans["loan_status"].str.strip().str.lower()   # Repaid / repaid / REPAID -> repaid
print("exact duplicate rows:", loans.duplicated().sum())

closed = loans["closed_date"].notna()
loans["repaid"] = (closed & (loans["closed_date"] <= loans["due_date"])).astype(int)

d = loans.drop_duplicates("loan_id").copy()      # 42,000 unique loans
asof = d["closed_date"].max()
print("unique loans:", len(d), "| data as of:", asof.date())

summary = pd.DataFrame({
    "check": ["Repaid on/before due % (all)", "Repaid on/before due % (already due)",
              "Closed before disbursed", "Missing disbursement_date",
              "Missing due_date", "Missing closed_date"],
    "value": [round(d["repaid"].mean() * 100, 1),
              round(d.loc[d["due_date"] <= asof, "repaid"].mean() * 100, 1),
              int((d["closed_date"] < d["disbursement_date"]).sum()),
              int(d["disbursement_date"].isna().sum()),
              int(d["due_date"].isna().sum()),
              int(d["closed_date"].isna().sum())],
})
display(summary)
display(d["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="n"))


exact duplicate rows: 168
unique loans: 42000 | data as of: 2026-11-04


,check,value
0,Repaid on/before due % (all),81.6
1,Repaid on/before due % (already due),83.6
2,Closed before disbursed,0.0
3,Missing disbursement_date,1235.0
4,Missing due_date,0.0
5,Missing closed_date,3772.0


,status,n
0,repaid,38228
1,written off,1513
2,overdue,1229
3,active,1030


In [4]:
# 3. Payments: keep unique successful payments, then amount collected per loan
pay = data["payments"].copy()
pay["amount_gbp"] = pd.to_numeric(pay["amount_gbp"], errors="coerce")
pay["payment_status"] = pay["payment_status"].str.strip().str.lower()
print("duplicate payment_ids:", pay["payment_id"].duplicated().sum())
display(pay["payment_status"].value_counts(dropna=False).reset_index())

OK_STATUS = ["success"]
ok = pay.drop_duplicates("payment_id")
ok = ok[ok["payment_status"].isin(OK_STATUS)]
paid = ok.groupby("loan_id")["amount_gbp"].sum().rename("paid_gbp")

d = d.drop(columns=["paid_gbp"], errors="ignore").merge(paid, on="loan_id", how="left")
d["paid_gbp"] = d["paid_gbp"].fillna(0)

due_now = d[d["due_date"] <= asof]
print("Collected / due (GBP), loans already due:",
      round(due_now["paid_gbp"].sum() / due_now["total_repayable_gbp"].sum() * 100, 1), "%")
print("Fully paid (>= 99% of repayable):",
      round((due_now["paid_gbp"] >= 0.99 * due_now["total_repayable_gbp"]).mean() * 100, 1), "%")


duplicate payment_ids: 277


,payment_status,count
0,success,40001
1,failed,6505


Collected / due (GBP), loans already due: 92.3 %
Fully paid (>= 99% of repayable): 93.3 %


In [5]:
# 4. Payment failures: by processor, method and reason
pu = pay.drop_duplicates("payment_id").copy()
pu["ts"] = pd.to_datetime(pu["payment_datetime_utc"], errors="coerce", utc=True).dt.tz_convert(None)
pu["failed"] = pu["payment_status"] == "failed"

for col in ["processor", "payment_method"]:
    display(pu.groupby(col)["failed"].agg(attempts="size", fail_rate="mean")
              .round(3).sort_values("fail_rate", ascending=False))
display(pu.loc[pu["failed"], "failure_reason"].value_counts().reset_index())


,attempts,fail_rate
processor,,
PROC_B,25906,0.183
PROC_A,13280,0.130
FPS,7043,0.000


,attempts,fail_rate
payment_method,,
Card auto-debit,26772,0.230
In-app card payment,12414,0.027
Bank transfer,7043,0.000


,failure_reason,count
0,INSUFFICIENT_FUNDS,2582
1,PROCESSOR_TIMEOUT,1610
2,DO_NOT_HONOUR,1204
3,AUTH_ERROR,470
4,CARD_DECLINED,329
5,CARD_EXPIRED,282


In [6]:
# 5. Model table: de-duplicated loans that are already due
m = d[d["due_date"] <= asof].sort_values("due_date").reset_index(drop=True)

cust = data["customers"].drop_duplicates("customer_id")
cust_cols = [c for c in ["region", "device_os"] if c in cust.columns]
m = m.merge(cust[["customer_id"] + cust_cols], on="customer_id", how="left")

m["interest_ratio"] = m["total_repayable_gbp"] / m["principal_amount_gbp"] - 1
m["disb_month"] = m["disbursement_date"].dt.month
m["disb_dow"] = m["disbursement_date"].dt.dayofweek
m["due_month"] = m["due_date"].dt.to_period("M").astype(str)

print("On-time rate by due month")
display(m.groupby("due_month")["repaid"].agg(n="size", on_time="mean").round(3))


On-time rate by due month


,n,on_time
due_month,,
2025-10,139,0.935
2025-11,956,0.915
2025-12,1598,0.913
2026-01,2134,0.913
2026-02,2270,0.913
2026-03,3025,0.905
2026-04,3095,0.912
2026-05,3971,0.840
2026-06,4294,0.815


In [7]:
# 6. Effect of a failed payment on repayment
paid_ok = pu[pu["payment_status"] == "success"].groupby("loan_id")["amount_gbp"].sum()
n_fail = pu[pu["failed"]].groupby("loan_id").size()
first_reason = pu[pu["failed"]].sort_values("ts").groupby("loan_id")["failure_reason"].first()

m2 = m[["loan_id", "repaid", "due_date", "total_repayable_gbp"]].copy()
m2["any_fail"] = m2["loan_id"].map(n_fail).fillna(0) > 0
m2["fully_paid"] = m2["loan_id"].map(paid_ok).fillna(0) >= 0.99 * m2["total_repayable_gbp"]
m2["first_fail_reason"] = m2["loan_id"].map(first_reason)

print("Loans with vs without a failed payment")
display(m2.groupby("any_fail")[["repaid", "fully_paid"]].agg(["size", "mean"]).round(3))
print("By first failure reason")
display(m2.groupby("first_fail_reason")[["repaid", "fully_paid"]].mean().round(3)
        .join(m2.groupby("first_fail_reason").size().rename("loans")))

# What changed in May 2026?
pu["month"] = pu["ts"].dt.to_period("M").astype(str)
display(pu.groupby("month").agg(attempts=("failed", "size"), fail_rate=("failed", "mean")).round(3))
display(pu.pivot_table(index="month", columns="processor", values="failed", aggfunc="mean").round(3))
display(pu.pivot_table(index="month", columns="payment_method", values="failed", aggfunc="size"))


Loans with vs without a failed payment


repaid        fully_paid       
           size   mean       size   mean
any_fail                                
False     35382  0.940      35382  0.972
True       5567  0.169       5567  0.686

By first failure reason


,repaid,fully_paid,loans
first_fail_reason,,,
AUTH_ERROR,0.209,0.749,470
CARD_DECLINED,1.000,1.000,329
CARD_EXPIRED,0.000,0.592,282
DO_NOT_HONOUR,0.141,0.687,1204
INSUFFICIENT_FUNDS,0.000,0.563,1672
PROCESSOR_TIMEOUT,0.215,0.745,1610


,attempts,fail_rate
month,,
2025-10,160,0.056
2025-11,1074,0.102
2025-12,1812,0.124
2026-01,2445,0.114
2026-02,2533,0.114
2026-03,3448,0.119
2026-04,3503,0.105
2026-05,4427,0.146
2026-06,4841,0.148


processor,FPS,PROC_A,PROC_B
month,,,
2025-10,0.0,0.071,NaN
2025-11,0.0,0.120,NaN
2025-12,0.0,0.145,NaN
2026-01,0.0,0.133,0.000
2026-02,0.0,0.132,0.000
2026-03,0.0,0.140,0.000
2026-04,0.0,0.122,0.000
2026-05,0.0,0.082,0.186
2026-06,0.0,0.000,0.175


payment_method,Bank transfer,Card auto-debit,In-app card payment
month,,,
2025-10,33,84,43
2025-11,156,665,253
2025-12,264,1114,434
2026-01,341,1494,610
2026-02,330,1583,620
2026-03,488,2164,796
2026-04,486,2198,819
2026-05,718,2495,1214
2026-06,748,2711,1382


In [8]:
# 7. Model v1 (baseline): loan terms, region, device. Time-based split.
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

num = ["term_days", "principal_amount_gbp", "total_repayable_gbp",
       "interest_ratio", "disb_month", "disb_dow"]
cat = [c for c in ["repayment_method"] + cust_cols if c in m.columns]
X = pd.get_dummies(m[num + cat], columns=cat, dtype=int)
y = m["repaid"]

cut = int(len(m) * 0.75)             # oldest 75% train, newest 25% test
Xtr, Xte, ytr, yte = X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]

lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
gb = HistGradientBoostingClassifier(random_state=42)
lr.fit(Xtr, ytr)
gb.fit(Xtr, ytr)

print("train rows:", len(Xtr), "| test rows:", len(Xte), "| test on-time rate:", round(yte.mean() * 100, 1), "%")
display(pd.DataFrame({
    "model": ["Logistic", "GradientBoosting"],
    "AUC": [round(roc_auc_score(yte, lr.predict_proba(Xte)[:, 1]), 3),
            round(roc_auc_score(yte, gb.predict_proba(Xte)[:, 1]), 3)],
}))

res = m.iloc[cut:].copy()
res["p"] = gb.predict_proba(Xte)[:, 1]
for g in cust_cols:
    print("Fairness v1 by", g)
    display(res.groupby(g).agg(n=("repaid", "size"), actual=("repaid", "mean"),
                               predicted=("p", "mean")).round(3))


train rows: 30711 | test rows: 10238 | test on-time rate: 78.3 %


,model,AUC
0,Logistic,0.569
1,GradientBoosting,0.560


Fairness v1 by region


,n,actual,predicted
region,,,
East Midlands,705,0.796,0.795
East of England,606,0.774,0.808
London,1728,0.791,0.797
North East,429,0.811,0.792
North West,1319,0.758,0.789
Northern Ireland,489,0.744,0.784
Scotland,805,0.783,0.800
South East,1065,0.787,0.784
South West,737,0.767,0.794


Fairness v1 by device_os


,n,actual,predicted
device_os,,,
Android,5228,0.783,0.797
iOS,5010,0.783,0.791


In [9]:
# 8. Model v2: recent loans only (due from May 2026) + customer features
from sklearn.inspection import permutation_importance

c2 = data["customers"].drop_duplicates("customer_id").copy()
c2["age"] = 2026 - pd.to_numeric(c2["year_of_birth"], errors="coerce")
c2["payday"] = pd.to_numeric(c2["payday_day_of_month"], errors="coerce")
c2["signup_date"] = pd.to_datetime(c2["signup_date"], errors="coerce")
keep = ["customer_id", "age", "payday", "signup_date", "employment_status",
        "monthly_income_band", "acquisition_channel", "sms_opt_in", "phone_valid"]

r = m.merge(c2[keep], on="customer_id", how="left")
r["customer_loan_number"] = pd.to_numeric(r["customer_loan_number"], errors="coerce")
r["days_since_payday"] = (r["due_date"].dt.day - r["payday"]) % 30
r["tenure_days"] = (r["disbursement_date"] - r["signup_date"]).dt.days
r = r[r["due_date"] >= "2026-05-01"].sort_values("due_date").reset_index(drop=True)

num2 = ["term_days", "principal_amount_gbp", "interest_ratio", "customer_loan_number",
        "age", "days_since_payday", "tenure_days"]
cat2 = ["repayment_method", "region", "device_os", "employment_status",
        "monthly_income_band", "acquisition_channel", "sms_opt_in", "phone_valid"]
X2 = pd.get_dummies(r[num2 + cat2], columns=cat2, dtype=int)
y2 = r["repaid"]

cut2 = int(len(r) * 0.75)
Xtr2, Xte2, ytr2, yte2 = X2.iloc[:cut2], X2.iloc[cut2:], y2.iloc[:cut2], y2.iloc[cut2:]
gb2 = HistGradientBoostingClassifier(random_state=42).fit(Xtr2, ytr2)
print("rows:", len(r), "| test on-time rate:", round(yte2.mean() * 100, 1), "%",
      "| AUC:", round(roc_auc_score(yte2, gb2.predict_proba(Xte2)[:, 1]), 3))

pi = permutation_importance(gb2, Xte2, yte2, scoring="roc_auc", n_repeats=3, random_state=42)
print("Top features")
display(pd.Series(pi.importances_mean, index=Xte2.columns).sort_values(ascending=False).head(10).round(4))


rows: 27732 | test on-time rate: 78.3 % | AUC: 0.604
Top features


repayment_method_Card auto-debit        0.0400
principal_amount_gbp                    0.0289
sms_opt_in_N                            0.0153
phone_valid_N                           0.0143
days_since_payday                       0.0104
employment_status_Employed full-time    0.0089
age                                     0.0020
employment_status_Unemployed            0.0016
tenure_days                             0.0015
interest_ratio                          0.0014
dtype: float64

In [10]:
# 9. Fairness check for model v2: predicted vs actual by group
res2 = r.iloc[cut2:].copy()
res2["p"] = gb2.predict_proba(Xte2)[:, 1]
res2["age_band"] = pd.cut(res2["age"], [0, 24, 34, 44, 54, 120],
                          labels=["<25", "25-34", "35-44", "45-54", "55+"])
for g in ["age_band", "employment_status", "monthly_income_band", "region"]:
    print("Fairness v2 by", g)
    display(res2.groupby(g, observed=True).agg(n=("repaid", "size"), actual=("repaid", "mean"),
                                               predicted=("p", "mean")).round(3))


Fairness v2 by age_band


,n,actual,predicted
age_band,,,
<25,1254,0.763,0.781
25-34,2446,0.789,0.805
35-44,2270,0.780,0.806
45-54,827,0.799,0.809
55+,136,0.816,0.820


Fairness v2 by employment_status


,n,actual,predicted
employment_status,,,
Employed full-time,3107,0.812,0.811
Employed part-time,720,0.753,0.796
Gig / contract,1071,0.753,0.792
Self-employed,1329,0.777,0.796
Student,286,0.727,0.800
Unemployed,296,0.743,0.777


Fairness v2 by monthly_income_band


,n,actual,predicted
monthly_income_band,,,
"Under £1,000",1529,0.770,0.795
"£1,000 - £1,999",2328,0.779,0.801
"£2,000 - £2,999",1862,0.809,0.805
"£3,000 and over",880,0.772,0.808


Fairness v2 by region


,n,actual,predicted
region,,,
East Midlands,475,0.785,0.808
East of England,410,0.768,0.802
London,1183,0.802,0.804
North East,311,0.794,0.808
North West,895,0.761,0.796
Northern Ireland,327,0.743,0.797
Scotland,524,0.788,0.808
South East,718,0.790,0.797
South West,488,0.736,0.795


In [11]:
r2 = r.copy()
r2["payday_bucket"] = pd.cut(r2["days_since_payday"], [-1, 4, 9, 14, 19, 24, 30],
                             labels=["0-4", "5-9", "10-14", "15-19", "20-24", "25-29"])
for col in ["sms_opt_in", "phone_valid", "repayment_method", "payday_bucket"]:
    print("On-time rate by", col)
    display(r2.groupby(col, observed=True)["repaid"].agg(n="size", on_time="mean").round(3))

On-time rate by sms_opt_in


,n,on_time
sms_opt_in,,
N,1642,0.704
Y,26090,0.805


On-time rate by phone_valid


,n,on_time
phone_valid,,
N,1936,0.717
Y,25796,0.806


On-time rate by repayment_method


,n,on_time
repayment_method,,
Bank transfer,3910,0.850
Card auto-debit,17410,0.773
In-app card payment,6412,0.840


On-time rate by payday_bucket


,n,on_time
payday_bucket,,
0-4,2160,0.845
5-9,2692,0.821
10-14,2160,0.824
15-19,1794,0.814
20-24,1948,0.789
25-29,1855,0.745


In [12]:
# contactable and not on card auto-debit vs not contactable and on card auto-debit
r2["contactable"] = (r2["sms_opt_in"] == "Y") & (r2["phone_valid"] == "Y")
display(r2.groupby(["contactable", "repayment_method"])["repaid"].agg(n="size", on_time="mean").round(3))

n  on_time
contactable repayment_method                   
False       Bank transfer          476    0.706
            Card auto-debit       2245    0.729
            In-app card payment    747    0.669
True        Bank transfer         3434    0.870
            Card auto-debit      15165    0.780
            In-app card payment   5665    0.862

In [13]:
od = d[d["status"] == "overdue"].copy()
od = od.merge(c2[["customer_id", "sms_opt_in", "phone_valid", "payday", "employment_status"]],
              on="customer_id", how="left")
od["outstanding_gbp"] = (od["total_repayable_gbp"] - od["paid_gbp"]).clip(lower=0)
od["last_fail"] = od["loan_id"].map(
    pu[pu["failed"]].sort_values("ts").groupby("loan_id")["failure_reason"].last())
od["contactable"] = (od["sms_opt_in"] == "Y") & (od["phone_valid"] == "Y")

def action(row):
    f = row["last_fail"]
    if f in ("PROCESSOR_TIMEOUT", "AUTH_ERROR"):
        return "1 Retry card today"
    if f == "INSUFFICIENT_FUNDS":
        return "2 Retry the day after payday"
    if f == "DO_NOT_HONOUR":
        return "3 Retry in 2-3 days + bank transfer link"
    if f == "CARD_EXPIRED":
        return "4 Ask to update card / bank transfer link"
    if not row["contactable"]:
        return "5 Email or in-app contact (no SMS)"
    return "6 SMS reminder + bank transfer link"

od["action"] = od.apply(action, axis=1)

display(od.groupby("action").agg(loans=("loan_id", "size"),
                                 outstanding_gbp=("outstanding_gbp", "sum"),
                                 avg_days_past_due=("days_past_due", "mean")).round(0))

worklist = od.sort_values(["action", "outstanding_gbp"], ascending=[True, False])[
    ["loan_id", "customer_id", "action", "outstanding_gbp", "days_past_due", "payday", "last_fail"]]
worklist.to_csv("recovery_worklist.csv", index=False)
display(worklist.head(10))

,loans,outstanding_gbp,avg_days_past_due
action,,,
1 Retry card today,248,140128.0,45.0
2 Retry the day after payday,323,211416.0,46.0
3 Retry in 2-3 days + bank transfer link,178,107017.0,47.0
4 Ask to update card / bank transfer link,45,30028.0,37.0
5 Email or in-app contact (no SMS),103,65950.0,47.0
6 SMS reminder + bank transfer link,332,218017.0,48.0


,loan_id,customer_id,action,outstanding_gbp,days_past_due,payday,last_fail
477,L1033062,C117086,1 Retry card today,1837.5,49.0,25.0,PROCESSOR_TIMEOUT
53,L1025475,C109337,1 Retry card today,1807.0,83.0,NaN,PROCESSOR_TIMEOUT
1097,L1040416,C101600,1 Retry card today,1788.5,9.0,NaN,PROCESSOR_TIMEOUT
484,L1033129,C102778,1 Retry card today,1754.5,3.0,28.0,PROCESSOR_TIMEOUT
503,L1033381,C117020,1 Retry card today,1725.0,63.0,NaN,PROCESSOR_TIMEOUT
799,L1036860,C113457,1 Retry card today,1710.0,45.0,NaN,PROCESSOR_TIMEOUT
432,L1032696,C109840,1 Retry card today,1644.5,66.0,NaN,PROCESSOR_TIMEOUT
1055,L1039768,C110306,1 Retry card today,1568.0,12.0,25.0,PROCESSOR_TIMEOUT
259,L1030816,C111990,1 Retry card today,1392.0,23.0,NaN,PROCESSOR_TIMEOUT
712,L1035873,C107310,1 Retry card today,1380.3,69.0,31.0,AUTH_ERROR


## Roll-into-default model (scored on the day the loan is worked)

The brief asks us to predict who will **roll into default**, not who will pay on time. So the target here is:
*given a loan is already late, will it be written off?*

- **Training rows:** daily snapshots of resolved late loans at 1, 3, 5, 7, 10, 14 and 21 days past due. A snapshot uses only what was known on that day: due-date failures, payments since the due date, and customer and loan fields.
- **Split by time:** train on older loans, test on the newest 30%.
- **Model:** logistic regression. It tied with tuned gradient boosting, random forest and an ensemble in testing, and it is easier to explain. Age is excluded.
- The code lives in `core.py`, so the dashboard and this notebook use exactly the same logic.

In [ ]:
# 10. Recovery window: how long after the due date do late loans still get repaid?
import sys
sys.path.insert(0, "..")
import core

res = core.load()
m_, asof_ = res["m"], res["asof"]
late = m_[(m_["repaid"] == 0) & (m_["due_date"] <= asof_ - pd.Timedelta(days=core.RESOLVED_AFTER_DAYS))]
late_repaid = late[late["status"] != "written off"]
days_late = (late_repaid["closed_date"] - late_repaid["due_date"]).dt.days
print("Resolved late loans:", len(late), "| written off:", round((late["status"] == "written off").mean() * 100, 1), "%")
print("Days past due when late loans were finally repaid:")
display(days_late.describe(percentiles=[.5, .75, .9, .95]).round(1).to_frame("days_late"))
print(f"=> no late loan was repaid after {days_late.max()} days past due; "
      f"work window set to 1-{core.WORK_WINDOW_DPD} dpd, older loans go to the recovery track")

In [ ]:
# 11. Roll-into-default model: time-holdout performance and drivers
ev = res["model_eval"]
print(f"train loans: {ev['train_loans']} | test loans: {ev['test_loans']} (due after {ev['test_from'].date()})")
print(f"Test AUC: {ev['auc']:.3f}  (days past due alone: {ev['auc_dpd_only']:.3f}; "
      f"notebook model v2 on the old on-time target: 0.604)")
print("AUC within each day past due (ranking loans on the same day):")
display(ev["auc_by_dpd"].round(3).rename("AUC").to_frame().T)

lr_ = res["risk_model"]
names = lr_[0].get_feature_names_out()
coef = pd.Series(lr_[-1].coef_[0], index=names).sort_values()
print("Strongest drivers (standardised log-odds; + means more likely to be written off)")
display(pd.concat([coef.head(8), coef.tail(8)]).round(3).to_frame("coef"))

In [ ]:
# 12. Daily call list: which ordering reaches the most money that would otherwise be written off?
print(f"Calling the top {ev['capacity']:.0%} of newly overdue test loans (day 1 past due):")
display((ev["gbp_reached"] * 100).round(1).rename("% of later write-off £ reached").to_frame())

# Fairness check: predicted vs actual write-off rate by group on the test snapshots
te_ = ev["test"]
for g in ["employment_status", "monthly_income_band", "region"]:
    print("Fairness by", g)
    display(te_.groupby(g, observed=True).agg(n=("written_off", "size"), actual=("written_off", "mean"),
                                              predicted=("p", "mean")).round(3))

print("Today's prioritised call list (top 15):")
display(res["worklist"].head(15))
display(res["worklist"].groupby(["stage", "action"]).agg(
    loans=("loan_id", "size"), outstanding_gbp=("outstanding_gbp", "sum"),
    expected_loss_gbp=("expected_loss_gbp", "sum")).round(0))